# RAG 프로젝트 전체 과정 정리

이 노트북은 2026-05-31 기준으로 retrieval, generation, PEFT, serving까지 진행한 전체 흐름과 시행착오를 정리한 문서입니다. 실제 지표는 저장된 CSV/JSONL에서 다시 확인할 수 있도록 코드 셀도 함께 넣었습니다.

## 0. 최종 결론

- 최종 retrieval 후보는 **KURE + Chroma 690 + dense qdecomp/RRF + doc scoring + target-aware**입니다.
- 검색은 `hit@5=0.9633`, `nDCG@5=0.9215`까지 올라갔습니다.
- 생성은 source_store/evidence/context routing을 붙였지만 Phase3 task score는 `0.3097`로 아직 낮습니다.
- PEFT는 v2 adapter가 base 대비 valid loss를 약 50.3% 낮췄지만, 라벨 수가 부족해 최종 성능 보장은 어렵습니다.
- 서비스는 채팅형 UI/history 처리까지 연결했습니다.

In [ ]:
from pathlib import Path
import pandas as pd, json
ROOT = Path('..').resolve()
retrieval_csv = ROOT/'outputs/eval/light_retrieval_sweep_690_phase34_gold50/summary_metrics.csv'
context_csv = ROOT/'outputs/generation/context_sweep_690_50/context_sweep_690_50_summary.csv'
phase3_csv = ROOT/'outputs/eval/final_690_phase34_gold_qwen_openai_judge/phase3_domain_results.csv'
phase4_csv = ROOT/'outputs/eval/final_690_phase34_gold_qwen_openai_judge/phase4_llm_judge_results.csv'
pred_path = ROOT/'outputs/generation/final_690_phase34_gold_qwen/126_service_route_v3_nonbudget_patch_123_budget_50_eval_predictions.jsonl'
gold_path = ROOT/'eval/evaluation/data/rfp_domain_gold_sample.jsonl'


## 1. 사용 데이터

| 단계 | 파일 | 역할 |
|---|---|---|
| 최종 chunk | `data/processed/chunks_v2_690.jsonl` | 최신 690 corpus chunk |
| 최종 source store | `data/processed/source_store_v2_690.jsonl` | document_summary, fact_candidates, final 값 |
| 최종 index | `indexes/chroma_kure_v1_chunks_v2_690/` | Chroma vector DB |
| 평가 gold | `eval/evaluation/data/rfp_domain_gold_sample.jsonl` | Phase3/4 gold 50문항 |
| 최종 prediction | `outputs/generation/final_690_phase34_gold_qwen/126_service_route_v3_nonbudget_patch_123_budget_50_eval_predictions.jsonl` | 최종 judge 평가에 사용 |

## 2. Retrieval 실험과 최종 조합

초기에는 BM25, dense, hybrid, rerank, FAISS/Chroma를 비교했습니다. 이후 평가가 문서 단위라는 점에 맞춰 chunk top-k를 그대로 쓰지 않고, 더 넓은 후보에서 문서 단위로 정리하는 방식으로 바꿨습니다. 최종적으로 dense qdecomp/RRF + doc scoring + target-aware 조합이 가장 안정적이었습니다.

In [ ]:
retrieval = pd.read_csv(retrieval_csv)
retrieval[['variant','num_questions','hit_at_5','mrr_at_5','ndcg_at_5','doc_recall_at_5','multi_doc_recall_at_5']].head(8)


| variant | 문항수 | hit@5 | MRR@5 | nDCG@5 | doc_recall@5 | multi_doc_recall@5 |
| --- | --- | --- | --- | --- | --- | --- |
| 104_best_dense_qdecomp_docscore_targetaware | 50 | 0.9633 | 0.9300 | 0.9215 | 0.9633 | 0.9127 |
| 109_best_target50 | 50 | 0.9633 | 0.9300 | 0.9215 | 0.9633 | 0.9127 |
| 108_best_preserve5 | 50 | 0.9567 | 0.9300 | 0.9175 | 0.9567 | 0.8968 |
| 107_hybrid_docscore_targetaware | 50 | 0.9333 | 0.9400 | 0.9144 | 0.9333 | 0.8413 |
| 105_dense_keyword_rerank | 50 | 0.8767 | 0.9500 | 0.8710 | 0.8767 | 0.7063 |
| 106_hybrid_keyword_rerank | 50 | 0.8167 | 0.9083 | 0.8221 | 0.8167 | 0.5635 |

### Retrieval에서 좋았던 것

- query decomposition + RRF: 복합 질문에서 후보 문서를 넓게 확보.
- document scoring: chunk가 아니라 문서 단위 평가와 맞춤.
- target-aware 보정: 기관명/사업명 힌트를 이용해 엉뚱한 유사 문서를 줄임.
- 문서 단위 중복 제어: 같은 문서 chunk가 top-k를 독식하는 문제 완화.

### Retrieval에서 별로였던 것

- keyword rerank: 키워드 overlap이 강한데 실제 target과 다른 문서를 올리는 경우가 있었음.
- hybrid+keyword rerank: 이 데이터에서는 dense target-aware보다 낮음.
- raw top 문서 전부 보존: retrieval 점수는 올랐지만 generation 품질은 오르지 않음.

## 3. Evidence Recall 진단

검색이 좋아 보여도 생성이 안 되는 이유를 보기 위해 evidence recall을 별도 참고 지표로 만들었습니다. 핵심 발견은 **정답 문서는 찾았지만 정답 근거 chunk가 context에 없는 경우**였습니다.

| 설정 | evidence_recall@5 | evidence_hit@5 | context_evidence_recall | doc_recall@5 | doc hit but evidence missed | 해석 |
| --- | --- | --- | --- | --- | --- | --- |
| 초기 104 remap 진단 | 0.0200 |  |  | 0.8400 | 39 | 정답 문서는 찾았지만 근거 chunk가 거의 빠짐 |
| baseline_existing_context | 0.2096 | 0.6600 | 0.2146 | 0.8033 | 17 | 기존 context 기준 |
| existing_core_rr_source_final | 0.3496 | 0.8800 | 0.6849 | 0.7433 | 3 | generation 연결 후보로 가장 안정적 |
| strict_pack required-first adaptive | 0.3821 | 0.8800 | 0.8875 | 0.7433 | 3 | 균형형 후보. context recall은 높음 |

## 4. Generation / Context 실험

처음에는 retrieved chunk를 그대로 넣었지만 context가 길고 noisy했습니다. 그래서 source_store, context pruning, evidence hierarchy, question type routing, 금액 정규화, source_store final 값 검증을 차례로 실험했습니다.

In [ ]:
context = pd.read_csv(context_csv)
context[['variant','phase3_task_score','budget_numeric_accuracy','required_field_accuracy','unanswerable_refusal_accuracy','hit_at_5','ndcg_at_5','multi_doc_recall_at_5']]


| variant | Phase3 task | budget | required field | unanswerable | hit@5 | nDCG@5 | multi_doc_recall@5 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| previous_final_target_evidence_50 | 0.3097 | 0.2083 | 0.1656 | 0.5500 | 0.9000 | 0.8741 | 0.7619 |
| new_preserve_top_50 | 0.3033 | 0.2083 | 0.1111 | 0.6000 | 0.9633 | 0.9215 | 0.9127 |

### Generation에서 배운 점

- source_store는 유용하지만 무조건 넣으면 noisy해집니다.
- `final_budget`, `final_period`는 retrieved chunk에서 같은 값이 확인될 때 더 안전합니다.
- 예산 질문은 원문 금액과 정규화 금액을 같이 넘겨야 합니다.
- required_fields/submission 질문은 요약보다 원문 필드명과 표 항목이 더 중요합니다.
- OpenAI Judge Phase4는 evidence-only 기준이라, gold 정답과 맞는지는 Phase3와 같이 봐야 합니다.

## 5. 실제 문항 사례

아래는 대표 문항에서 생성 답변과 gold를 나란히 본 것입니다. `Q003`은 source_store/evidence 값이 gold와 충돌한 사례이고, `Q029`는 금액 정규화/계산형 템플릿이 효과가 있었던 사례입니다.

| ID | 판정 | 유형 | 질문 | 생성 답변(정리) | 정답/Gold 요약 | Phase3 | Phase4 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| P3-SUB-004 | 미흡 | submission_eligibility_deadline | 경기도 포천시의 2024년 포천시 스마트도시 솔루션 확산사업 입찰참가자격은 무엇입니까? | 자격요건을 제대로 추출하지 못했습니다. | 정답 문서: 경기도 포천시_2024년 포천시 스마트도시 솔루션 확산사업 (협상에 의한 .hwp \| 제출서류: 제안서, 가격제안서, 산출내역서, 위임장, 재직증명서, 제안요약서, 발표자료, 사업자등록증, 청렴계약, 서약서, 확인서 \| 자격요건: 5) 조달청 입찰참가자격… | 0.000 | 4.500 |
| P3-SUB-006 | 미흡 | submission_eligibility_deadline | 경상북도 경주시의 'AI기반 실시간 신호제어시스템 구축 실증사업' 입찰에 참여하기 위해 필요한 주요 자격요건은 무엇입니까? | 자격요건은 문서에서 명시되지 않았으므로 제공된 정보 기반으로 판단할 수 없습니다. | 정답 문서: 경상북도 경주시_AI기반 실시간 신호제어시스템 구축 실증사업.hwp \| 제출서류: 제안서, 입찰참가신청서, 사업자등록증, 청렴계약, 서약서, 가격제안서, 확인서, 신용평가등급확인서, 발표자료 \| 자격요건: ○ 「중소기업제품 구매 촉진 및 판로지원에 관한 … | 0.000 | 4.250 |
| P3-SUB-010 | 미흡 | submission_eligibility_deadline | 대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 사업의 제출서류, 입찰참가자격, 입찰마감일은 무엇입니까? | 제출서류: 실적증명서, 제안서, 가격제안서, 사업자등록증, 소프트웨어사업자, 입찰참가신청서, 보안서약서, 서약서 / 마감일정: 2024-09-13 | 정답 문서: 대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp \| 제출서류: 실적증명서, 제안서, 가격제안서, 사업자등록증, 소프트웨어사업자, 입찰참가신청서, 보안서약서, 서약서 \| 자격요건: 2) 국가를 당사자로 하는 계약에 관한… | 0.000 | 2.500 |
| Q003 | 미흡 | budget | 코이카(KOICA) 전자조달의 '우즈베키스탄 열린 의정활동 상하원 국회 방송시스템 구축 및 지역의회 연계 개선 PMC 용역' 예산 규모는 어떻… | 예산은 2,000,000,000원입니다. | 정답 문서: KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp \| 예산: [긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 = 8000000000원 \| 합계/기준값: 8000000000원 \| identity_gold:… | 0.000 | 5.000 |
| Q005 | 미흡 | required_fields | 그랜드코리아레저(주)의 그룹웨어 구축 사업에서 목표로 하는 시스템 개선의 대상 범위를 모두 열거해 주십시오. | 주요 요구사항: 시스템 개선 대상 범위는 다음과 같습니다: - 조직 내부 업무 프로세스 자동화 - 사용자 권한 관리 및 보안 강화 - 문서 관리 및 공유 기능 개선 - 모바일 접근성 지원 - 통합 협업 환경 구축 … | 정답 문서: 그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역.hwp \| identity_gold: {"agencies": ["그랜드코리아레저(주)"], "project_names": ["2024년도 GKL 그룹웨어 시스템 구축 용역"]} \| req… | 0.000 | 3.700 |
| Q029 | 부분 | budget | 그랜드코리아레저(주)의 그룹웨어 시스템 구축 사업 예산(1,515,000,000원)과 부산국제영화제의 온라인서비스 재개발 사업(243,000,… | 두 사업의 예산 차이는 1,272,000,000원입니다. | 정답 문서: 그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역.hwp, (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp \| 예산: 2024년도 GKL 그룹웨어 시스템 구축 용역 = 1515000000원 \|… | 0.500 | 5.000 |
| Q004 | 부분 | required_fields | 한국가스공사의 ERP 구축 사업에 도입되는 신규 소프트웨어 내역과 인프라 내역을 모두 나열해 주시겠습니까? | 주요 요구사항: 한국가스공사의 [재공고]차세대 통합정보시스템(ERP) 구축 사업에 도입되는 신규 소프트웨어 및 인프라는 다음과 같습니다. - **신규 소프트웨어**: SAP ERP 인프라 도입 - **인프라 내용*… | 정답 문서: 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp \| identity_gold: {"agencies": ["한국가스공사"], "project_names": ["[재공고]차세대 통합정보시스템(ERP) 구축"]} \| required_field_… | 0.500 | 3.050 |
| Q025 | 부분 | required_fields | 한국수자원공사의 용인 첨단 시스템반도체 관련 용수공급사업이 목표로 하는 구체적인 총 공급량과 추진 단계에 대해 설명해 주세요. | 핵심 요약: 한국수자원공사의 용인 첨단 시스템반도체 국가산단 용수공급사업은 24년의 사업기간을 가진 용수공급사업으로, 용인 첨단 시스템반도체 국가산단의 수요에 대응하기 위한 용수공급을 목표로 하고 있습니다. 주요 … | 정답 문서: 한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp \| identity_gold: {"agencies": ["한국수자원공사"], "project_names": ["용인 첨단 시스템반도체 국가산단 용수공급사업 타당성"]} \| requi… | 0.500 | 3.250 |
| Q017 | 부분 | unanswerable | 고려대학교의 '차세대 포털·학사 정보시스템 구축사업'을 수주하기 위해서는 총괄 책임자가 반드시 정보보안기사 및 정보처리기사 자격증 원본을 필수… | 문서에서 확인할 수 없습니다. | 정답 문서: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf \| identity_gold: {"agencies": ["고려대학교"], "project_names": ["차세대 포털·학사 정보시스템 구축사업"]} \| unanswerable_gold: {"is_u… | 0.500 | 3.100 |
| Q018 | 부분 | unanswerable | 국립중앙의료원 2024년도 차세대 응급의료 상황관리시스템 구축 용역에서 제시된 콜센터 구축 시 필요한 상담원의 최소 채용 인원과 일일 목표 통… | 문서에서 제공된 정보는 콜센터 구축 시 필요한 상담원의 최소 채용 인원과 일일 목표 통화 횟수에 대한 구체적인 데이터를 포함하고 있지 않습니다. | 정답 문서: 국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.hwp \| identity_gold: {"agencies": ["국립중앙의료원"], "project_names": ["(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축"]} \|… | 0.500 | 2.200 |


In [ ]:
phase3 = pd.read_csv(phase3_csv)
phase4 = pd.read_csv(phase4_csv)
phase3[['id','question','task_family','phase3_task_score','budget_numeric_accuracy','required_field_accuracy','multi_doc_structure_score']].head(12)


## 5-1. 원래 틀렸던 문제가 개선된 사례

비교 기준은 `105_budget_priority_guarded_source`, 최종 결과는 `126_service_route_v3_nonbudget_patch_123_budget`입니다. 같은 question_id에서 Phase3 점수가 오른 문항입니다.

| ID | 개선 | 유형 | 점수 변화 | 질문 | 이전 답변 | 최종 답변 | 좋아진 이유 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Q068 | 완전 개선 | budget | 0.250 → 1.000 | 그랜드코리아레저(주) 그룹웨어 구축 예산(1,515,000,000원)과 인천광역시의 도시계획 통합관리시스템 예산(150,000,000원)을 통… | 인천광역시의 도시계획 통합관리시스템 예산은 150,000,000원입니다. 그러나 '그랜드코리아레저(주) 그룹웨어 구축' 예산에 대한 정보는 제공된 Context에 포함되지 않았습니다. 두 예산을 통합하여 발주할 경우 전체 예산을 계산하려면 '그랜드코리아레저(주) 그룹웨어 구축' 예산 정보가 추가로 필요합니다. | 예산: 1,515,000,000원 + 150,000,000원 = 1,665,000,000원 핵심 요약: 제공된 문서에 따르면 '그랜드코리아레저(주)' 그룹웨어 구축 예산은 1,515,000,000원이고, 인천광역시의 도시계획 통합관리시스템 예산은 150,000,000원입니다. 두 예산을 합산하면 총 1,665,000,000원입니다. | 예산 fact_type 우선 + source_store final_budget 검증 + 금액 정규화 |
| Q025 | 부분 개선 | required_fields | 0.250 → 0.500 | 한국수자원공사의 용인 첨단 시스템반도체 관련 용수공급사업이 목표로 하는 구체적인 총 공급량과 추진 단계에 대해 설명해 주세요. | 용인 첨단 시스템반도체 국가산단 용수공급사업의 총 공급량은 2,392,940,000원 규모이며, 이 사업은 24년 기간 동안 진행됩니다. 핵심 요약으로는 용수공급사업의 타당성 검토 및 수자원공사의 발주기관으로서의 역할이 강조됩니다. | 핵심 요약: 한국수자원공사는 용인 첨단 시스템반도체 국가산단 용수공급사업을 통해 특정 용수공급량을 목표로 하고 있으며, 이 사업의 예산은 2,392,940,000원입니다. 다만, 구체적인 총 공급량이나 추진 단계에 대한 명확한 정보는 제공된 문서에서 확인되지 않았습니다. | required field 전용 context + target evidence 보정 |
| Q042 | 부분 개선 | required_fields | 0.000 → 0.250 | 고려대학교의 '차세대 포털·학사 정보시스템 구축사업'의 근본적인 추진 배경 중 하나로 꼽히는 대외적 교육환경 변화는 무엇인가요? | 대외적 교육환경 변화로 인해 기존의 전통적인 정보전달 방식이 한계를 보이고 있으며, 디지털 시대에 맞춘 통합적이고 실시간의 정보제공 체계가 필요하다는 점이 배경 중 하나로 제시되었다. | 대외적 교육환경 변화로 인해 기존의 학사 정보 시스템이 한계를 보이고 있으며, 디지털 전환 및 교육 혁신을 위한 차세대 포털·학사 정보시스템 구축이 필요하다는 배경이 있다. | required field 전용 context + target evidence 보정 |
| Q006 | 부분 개선 | required_fields | 0.000 → 0.250 | 국립중앙의료원의 차세대 응급의료 상황관리시스템 구축 위탁용역에서 응급실 섭외 지연 문제를 해결하기 위해 제시한 콜센터 구축 외에 함께 도입되는… | 문제 해결을 위한 통신 환경 조건은 제공된 문서에서 명시되지 않았습니다. | 문서에 제공된 정보는 응급실 섭외 지연 문제 해결을 위한 콜센터 구축 외에 통신 환경의 조건에 대한 구체적인 내용을 포함하고 있지 않습니다. | required field 전용 context + target evidence 보정 |
| Q033 | 부분 개선 | multi_doc_comparison | 0.222 → 0.444 | 고려대학교의 '학사 정보시스템', 국립중앙의료원의 '상황관리시스템', 그리고 인천광역시의 '통합관리시스템' 구축 사업에서 공통적으로 해결하고자… | 계산 과정: 1. 11,270,000,000원 + 1,400,000,000원 = 12,670,000,000원 따라서 합계는 12,670,000,000원입니다. | 계산 과정: 1. 11,270,000,000원 + 1,400,000,000원 = 12,670,000,000원 따라서 합계는 12,670,000,000원입니다. 예산: 12,670,000,000원 핵심 요약: 세 핵심 개선 대상 사업의 공통적인 한계점은 기존 단절 운영 방식으로 인해 시스템 간 데이터 공유 및 통합 관리가 어려웠으며, 이를 통해 효율적인 운영과 정보 공유를 도모하고자 한다. | 문서 단위 selection + 다중 문서별 evidence 유지 + target-aware 보정 |
| Q072 | 부분 개선 | multi_doc_comparison | 0.167 → 0.333 | 고려대학교 학사 정보시스템 사업과 부산국제영화제 IT 온라인서비스 재개발 사업이 모두 유저의 편의성 및 대외 이미지 제고를 다룹니다. 양측 조… | 고려대학교 학사 정보시스템 사업과 부산국제영화제 IT 온라인서비스 재개발 사업은 각각의 최종 사용자를 다르게 설정하고 있습니다. 고려대학교의 경우, 학생, 교수, 관리자 등 교내 구성원이 주요 사용자이며, 이들은 학사 정보의 효율적 관리를 통해 대외 이미지를 제고하고자 합니다. 반면, 부산국제영화제의 IT 온라인서비스는 영화제 참여자, 관객, 협력업체 등 외부 사용자를 중심으로 하며, 이들의 편의성을 개선하여 영화제의 국제적 위상을 강화하는 데 목적이 있습니다. | 고려대학교 학사 정보시스템 사업과 부산국제영화제 IT 온라인서비스 재개발 사업은 각각의 최종 사용자를 다르게 설정하고 있습니다. 고려대학교의 경우, 학생, 교수, 관리자 등 교내 구성원이 주요 사용자이며, 이들은 학사 정보의 효율적 관리를 통해 대외 이미지 제고를 목표로 합니다. 반면, 부산국제영화제의 경우, 영화제 방문객, 영화 제작진, 미디어 기자 등 외부 사용자가 주요 대상이며, 이들은 온라인 서비스를 통해 영화제의 대외적 영향력을 확대하고자 합니다. | 문서 단위 selection + 다중 문서별 evidence 유지 + target-aware 보정 |
| Q031 | 부분 개선 | multi_doc_comparison | 0.167 → 0.333 | 국립중앙의료원의 위탁용역과 그랜드코리아레저(주)의 그룹웨어 구축 용역에서 각각 핵심으로 다루는 정보시스템의 업무 목적과 범위를 비교해 주십시오. | 국립중앙의료원의 위탁용역과 그랜드코리아레저(주)의 그룹웨어 구축 용역은 서로 다른 목적과 범위를 가지고 있습니다. 국립중앙의료원의 위탁용역은 의료 서비스 운영 및 관리를 위한 시스템을 중심으로 하고 있으며, 이는 의료기관의 효율적인 운영을 지원하기 위한 정보시스템입니다. 반면, 그랜드코리아레저(주)의 그룹웨어 구축 용역은 회사 내부의 협업 및 정보 공유를 위한 통합 시스템을 목표로 하며, 조직 내 다양한 부서 간의 협업을 강화하는 데 초점을 맞추고 있습니다. | 국립중앙의료원의 위탁용역과 그랜드코리아레저(주)의 그룹웨어 구축 용역은 서로 다른 목적과 범위를 가지고 있습니다. 국립중앙의료원의 위탁용역은 응급의료 상황관리시스템 구축을 통해 의료 서비스의 효율성과 대응 능력을 강화하는 것이 주요 목표이며, 이는 의료 정보의 실시간 관리와 협업을 위한 시스템을 제공하는 데 중점을 두고 있습니다. 반면, 그랜드코리아레저(주)의 그룹웨어 구축 용역은 조직 내부의 커뮤니케이션과 협업을 지원하기 위해 통합적인 정보 관리 시스템을 구… | 문서 단위 selection + 다중 문서별 evidence 유지 + target-aware 보정 |
| Q011 | 부분 개선 | multi_doc_comparison | 0.167 → 0.333 | 코이카 전자조달의 우즈베키스탄 방송시스템 사업과 아시아물위원회 사무국의 스마트 관개시스템 사업의 구축 대상 국가와 분야를 비교 설명해 주십시오. | 코이카 전자조달의 우즈베키스탄 ICT기반의 수자원정보 사업과 사단법인아시아물위원회사무국의 우즈벡-키르기즈스탄 기후변화대응 스 사업의 구축 대상 국가와 분야는 다음과 같습니다. 1. **KOICA 전자조달 - 우즈베키스탄 ICT기반의 수자원정보 사업** - 대상 국가: 우즈베키스탄 - 분야: 수자원 정보 시스템 구축 및 운영 2. **사단법인아시아물위원회사무국 - 우즈벡-키르기즈스탄 기후변화대응 스 사업** - 대상 국가: 우즈벡 및 키르기즈스탄 - 분야: 기후… | 코이카 전자조달의 우즈베키스탄 방송시스템 사업과 아시아물위원회 사무국의 스마트 관개시스템 사업은 모두 우즈베키스탄을 대상으로 하지만, 구축 분야가 다릅니다. 첫 번째 사업은 방송시스템 관련이며, 두 번째 사업은 스마트 관개시스템(수자원 관리) 관련입니다. | 문서 단위 selection + 다중 문서별 evidence 유지 + target-aware 보정 |
| Q034 | 부분 개선 | multi_doc_comparison | 0.278 → 0.389 | 코이카의 '우즈베키스탄 방송시스템 사업', 아시아물위원회의 '우즈벡-키르기즈스탄 관개시스템 사업', 수출입은행의 '모잠비크 ITS 타당성 조사… | 예산: 한국수출입은행 모잠비크 ITS 타당성 조사 사업의 예산은 1,247,000,000원입니다. 핵심 요약: 해당 사업은 모잠비크 마푸토 지역의 지능형교통시스템(ITS) 구축을 위한 타당성 조사 사업이며, 한국수출입은행이 발주기관입니다. | 예산: - 아시아물위원회 우즈벡-키르기즈스탄 관개시스템 사업: 5,031,000,000원 - 수출입은행 모잠비크 ITS 타당성 조사 사업: 1,247,000,000원 핵심 요약: - 아시아물위원회 우즈벡-키르기즈스탄 관개시스템 사업: 관개시스템 구축을 위한 기후변화 대응 프로젝트 - 수출입은행 모잠비크 ITS 타당성 조사 사업: 지능형교통시스템(ITS) 타당성 조사 및 구축 지원 | 문서 단위 selection + 다중 문서별 evidence 유지 + target-aware 보정 |
| Q054 | 부분 개선 | multi_doc_comparison | 0.222 → 0.333 | 한국가스공사의 '차세대 통합정보시스템', 나노종합기술원의 '설비온라인 시스템', 그랜드코리아레저의 '그룹웨어 시스템' 3개 인프라 전산화 구축… | 한국가스공사 차세대 통합정보시스템의 사업예산 근거를 context에서 확인할 수 없어 계산할 수 없습니다. | 예산: 한국가스공사 차세대 통합정보시스템은 1,515,000,000원을 기록하며, 이는 세 기관 중 가장 낮은 예산입니다. 핵심 요약: 세 기관은 인프라 전산화를 통해 업무 효율성 향상, 데이터 통합, 운영 비용 절감 등의 궁극적 개선 효과를 추구하고 있습니다. | 문서 단위 selection + 다중 문서별 evidence 유지 + target-aware 보정 |


## 6. PEFT 실험

PEFT는 Qwen3 8B 4bit에 QLoRA adapter를 붙이는 방식으로 진행했습니다. v2 adapter는 validation loss를 크게 낮췄지만, 학습 데이터가 50개라 사실성까지 안정화하기에는 부족했습니다. 추가 라벨 확장은 답변불가/부분답변 비중이 커져 최종 채택하지 않았습니다. 현재 best는 `outputs/peft/qwen3_8b_qlora_sft_v2_truncfix/adapter`입니다.

| 실험 | 결과 | 해석 |
| --- | --- | --- |
| Base Qwen valid loss | 1.5579 | 비교 기준 |
| PEFT v2 adapter valid loss | 0.7741 | 약 50.3% 개선 |
| v5 추가 정답형 라벨 | 13개 | 총 63개, 추가 학습 신호 부족 |
| 현재 서비스 adapter | qwen3_8b_qlora_sft_v2_truncfix/adapter | 현재 best |

## 7. Serving

최종 서비스는 `scripts/rag_service_web.py`입니다. 실험용 메타 패널을 제거하고 ChatGPT처럼 질문/답변 중심으로 보이게 만들었습니다. Enter 전송, Shift+Enter 줄바꿈, 첫 응답 로딩 안내, 생성 중 표시, 히스토리 기반 후속 질문 처리를 적용했습니다.

실행 예시:

```bash
.venv/bin/python scripts/rag_service_web.py --host 127.0.0.1 --port 7860 --use-best-adapter
```

## 8. 최종 조합

1. Retrieval: KURE + Chroma 690 + dense qdecomp/RRF + doc scoring + target-aware
2. Evidence: source_store/fact_type 기반 core evidence selection + required fact 우선
3. Context: source_store 조건부 사용, final 값은 retrieved chunk 확인 시 더 신뢰
4. Generation: Qwen3 8B 4bit + PEFT v2 adapter
5. Service: `rfp_service_route_v3` + history state + concise answer postprocess

## 9. 남은 일

- Phase3에서 낮은 budget/required_fields를 우선 개선.
- source_store final 값과 gold가 충돌하는 문항을 데이터 품질 이슈로 분리.
- 정답형 PEFT 라벨 50~100개를 추가 확보한 뒤 PEFT 재개.
- 서비스에는 출처/근거 보기를 옵션으로 붙이되 기본 답변은 짧게 유지.